In [ ]:
import arff
import pandas as pd
import pandas as pd
import numpy as np

# Load the ARFF file using the new library
with open('dataset.arff.txt', 'r') as f:
    dataset = arff.load(f)

# Extract column names from the metadata
columns = [attr[0] for attr in dataset['attributes']]

# Convert to a pandas DataFrame
df = pd.DataFrame(dataset['data'], columns=columns)

df2= pd.read_csv("test.csv")

time_cols = ['year', 'month', 'day', 'hour']

# --- normalise dtypes: liac-arff returns floats, read_csv returns ints ---
for d in (df, df2):
    for c in time_cols:
        d[c] = d[c].astype(int)
    d['station'] = d['station'].astype(str).str.strip()

df['dt']  = pd.to_datetime(df[time_cols])
df2['dt'] = pd.to_datetime(df2[time_cols])

# the hour we are actually predicting
df2['target_dt'] = df2['dt'] + pd.Timedelta(hours=1)

truth = df[['station', 'dt', 'PM2.5']].rename(
    columns={'dt': 'target_dt', 'PM2.5': 'pm25_next_hour'})

result_df = df2.merge(truth, on=['station', 'target_dt'], how='left')

# --- sanity checks ---
print(f"rows      : {len(df2)} -> {len(result_df)}")
print(f"unmatched : {result_df['pm25_next_hour'].isna().sum()}")
print(f"dup keys  : {df.duplicated(['station'] + time_cols).sum()}")
print(f"df range  : {df['dt'].min()}  ->  {df['dt'].max()}")
print(f"df2 range : {df2['dt'].min()}  ->  {df2['dt'].max()}")

# spot check: these two PM2.5 values must be identical
s = df2['station'].iloc[0]
r = result_df[result_df['station'] == s].iloc[0]
print(f"\nspot check [{s}]  dt={r['dt']}  target_dt={r['target_dt']}  "
      f"merged={r['pm25_next_hour']}")
print(df.loc[(df['station'] == s) & (df['dt'] == r['target_dt']),
             ['dt', 'PM2.5']].to_string(index=False))

# --- score ---
preds_df = pd.read_csv('submission.csv')
scored = result_df.merge(preds_df, on='id', how='inner')
valid  = scored.dropna(subset=['pm25_next_hour', 'PM2_5_next_hour'])
rmse   = np.sqrt(np.mean((valid['pm25_next_hour'] - valid['PM2_5_next_hour']) ** 2))

print(f"\nRMSE: {rmse:.4f}  over {len(valid)} of {len(scored)} rows "
      f"({len(scored) - len(valid)} dropped)")

FileNotFoundError: [Errno 2] No such file or directory: 'test(1).csv'